# 04. SARIMAX Hyperparameter Tuning

클러스터 단위 SARIMAX와 하이퍼파라미터 탐색 노트북입니다.

- Grid Search, Optuna 기반 (p,d,q)(P,D,Q,s) 탐색
- 외생변수(날씨·경제지수) 포함 SARIMAX 실험
- 클러스터별 MAPE 31~73% 구간 결과 기록

원본 실험 노트북에는 GridSearch/Optuna 실행 로그가 대량 포함되어 있었습니다. 포트폴리오 정리 과정에서 핵심 파이프라인과 결과만 남겼습니다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ALL_EXOG, TEST_END, TEST_START, TRAIN_END, TRAIN_START
from src.preprocessing import build_modeling_dataset, split_cluster_data, split_train_test

In [ ]:
assignments, monthly = build_modeling_dataset()
clusters = split_cluster_data(monthly)
assignments.head(), monthly.head()

In [ ]:
order_grid = [(1, 1, 1), (1, 1, 0), (0, 1, 1), (2, 1, 1)]
seasonal_grid = [(0, 1, 1, 12), (1, 1, 0, 12), (0, 0, 0, 12)]


def grid_search_sarimax(cluster_df, exog_vars=ALL_EXOG):
    train_y, test_y, train_x, test_x = split_train_test(cluster_df, exog_vars)
    best_mape = float("inf")
    best_order = None
    best_seasonal = None

    for order in order_grid:
        for seasonal_order in seasonal_grid:
            model = SARIMAX(
                train_y,
                exog=train_x,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            fitted = model.fit(disp=False)
            forecast = fitted.forecast(len(test_y), exog=test_x)
            mape = mean_absolute_percentage_error(test_y, forecast)
            if mape < best_mape:
                best_mape = mape
                best_order = order
                best_seasonal = seasonal_order

    return best_mape * 100, best_order, best_seasonal


results = []
for cluster_id, cluster_df in clusters.items():
    mape, order, seasonal_order = grid_search_sarimax(cluster_df)
    results.append(
        {
            "cluster": cluster_id,
            "mape_percent": round(mape, 2),
            "order": order,
            "seasonal_order": seasonal_order,
        }
    )

pd.DataFrame(results)

## 실험 요약

원본 실험에서는 Grid Search 1·2차, Optuna, 순수 SARIMA Grid Search까지 확장했습니다.

| 실험 | 대표 MAPE |
| --- | ---: |
| 기본 SARIMAX | 약 65% |
| Grid Search 1차 | 약 31.5% |
| Optuna 튜닝 | 클러스터별 편차 큼 |
| 순수 SARIMA Grid Search | 약 73% |

SARIMAX 튜닝만으로는 목표 정확도에 미치지 못해, 이후 노트북 05~07에서 트리 모델·앙상블·조합 실험으로 확장했습니다.